In [1]:
import sys
from types import ModuleType

# 제거된 vertexai 모듈을 메모리에 가짜로 등록
dummy_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertex.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertex

dummy_llm_vertex = ModuleType("langchain_community.llms.vertexai")
dummy_llm_vertex.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = dummy_llm_vertex

In [6]:
import pandas as pd
from ragas import EvaluationDataset

eval_raw_data2 = {
    "user_input": [
        "도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?",
        "시험 기간 평일 도서관 마감 시간은 몇 시인가요?"
    ],
    "reference": [
        "주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
        "시험 기간 평일 마감 시간은 22시까지입니다."
    ],
    "retrieved_contexts": [
        [
            "중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
            "중앙도서관 매점 운영 안내: 평일 오전 9시부터 오후 6시까지 운영합니다."
        ],
        [
            "중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.",
            "도서관 분실물 보관소는 1층 로비 안내실에서 상시 운영됩니다."
        ]
    ]
}

df_eval = pd.DataFrame(eval_raw_data2)

ragas_dataset2 = EvaluationDataset.from_pandas(df_eval)

print(df_eval)
ragas_dataset2

                         user_input  \
0  도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?   
1       시험 기간 평일 도서관 마감 시간은 몇 시인가요?   

                                          reference  \
0  주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.   
1                         시험 기간 평일 마감 시간은 22시까지입니다.   

                                  retrieved_contexts  
0  [중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며,...  
1  [중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.,...  


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference'], len=2)

In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from ragas.metrics import ContextPrecision, ContextRecall
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

load_dotenv()

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="none",
    temperature=0
)

evaluator_llm = LangchainLLMWrapper(model, bypass_temperature=True)

metrics = [
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

evaluation_result = evaluate(
    dataset=ragas_dataset2,
    metrics=metrics,
    raise_exceptions=True
)


C:\Users\yonggyo\AppData\Local\Temp\ipykernel_11660\4251859516.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_11660\4251859516.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_11660\4251859516.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(ap

In [8]:
evaluation_result

{'context_precision': 1.0000, 'context_recall': 1.0000}

In [5]:
df_result = evaluation_result.to_pandas()

output_csv_path = "search_eval_report.csv"
df_result.to_csv(output_csv_path, index=False, encoding="utf-8-sig")